In [ ]:
import os
import glob
from dotenv import load_dotenv
from pathlib import Path
import gradio as gr
from openai import OpenAI

import tiktoken
import numpy as np
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go

from langchain_core.messages import SystemMessage, HumanMessage
from evaluation import test

In [ ]:
# Setting up

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = "gpt-4.1-nano"
openai = OpenAI()

DB_NAME = "vector_db"

In [ ]:
# Setup the retriever with Chroma vector store and OpenAI embeddings (embedded systems knowledge base)
embedding_function = OpenAIEmbeddings(model="text-embedding-3-large")
vector_db = Chroma(
    collection_name="docs",
    embedding_function=embedding_function,
    persist_directory=DB_NAME
)
retriever = vector_db.as_retriever(search_kwargs={"k": 5})

SYSTEM_PROMPT_TEMPLATE = """You are a helpful assistant for embedded systems. Use the following context (MCUs, boards, sensors, tools, prices) to answer the question as accurately as possible.

Context:
{context}

If you do not know the answer, say you don't know. Do not make up information.
"""

In [ ]:
llm = ChatOpenAI(temperature=0, model_name=MODEL)

In [ ]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
# How many tokens in all the documents?
# Build full text from chunks (run after chunking), or from loaded documents
docs_for_text = globals().get("chunks") or globals().get("documents")
if docs_for_text is None:
    # Load from knowledge-base so this cell can run in any order (works with files only, no subfolders)
    loader = DirectoryLoader("knowledge-base", glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    docs_for_text = loader.load()
entire_knowledge_base = "\n\n".join(doc.page_content for doc in docs_for_text)

encoding = tiktoken.encoding_for_model(MODEL)
tokens = encoding.encode(entire_knowledge_base)
token_count = len(tokens)
print(f"Total tokens for {MODEL}: {token_count:,}")

In [ ]:
# Load in everything in the knowledgebase using LangChain's loaders
# Loads from knowledge-base/ directly (works when there are only files, no subfolders)

loader = DirectoryLoader("knowledge-base", glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
documents = loader.load()
for doc in documents:
    doc.metadata["doc_type"] = doc.metadata.get("doc_type", "knowledge-base")

print(f"Loaded {len(documents)} documents")

In [ ]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

In [ ]:
# Pick an embedding model (same as implementation/answer.py for consistency)

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(DB_NAME):
    Chroma(persist_directory=DB_NAME, embedding_function=embeddings, collection_name="docs").delete_collection()

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=DB_NAME,
    collection_name="docs",
)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

In [ ]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

In [ ]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['doc_type'] for metadata in metadatas]
type_colors = {'products': 'blue', 'employees': 'green', 'contracts': 'red', 'company': 'orange', 'knowledge-base': 'purple'}
colors = [type_colors.get(t, 'gray') for t in doc_types]

In [ ]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

n_samples = vectors.shape[0]
perplexity = min(30, max(1, n_samples - 1))  # t-SNE requires perplexity < n_samples
tsne = TSNE(n_components=2, random_state=42, perplexity=perplexity)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [ ]:
# Let's try 3D!

n_samples = vectors.shape[0]
perplexity = min(30, max(1, n_samples - 1))  # t-SNE requires perplexity < n_samples
tsne = TSNE(n_components=3, random_state=42, perplexity=perplexity)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

In [ ]:
SYSTEM_PREFIX = """
You represent an embedded systems knowledge assistant.
You are an expert in answering questions about embedded systems: MCUs, development boards, sensors, debuggers, RTOS, connectivity modules, and related components (prices, specs, part numbers).
You are provided with additional context that might be relevant to the user's question.
Give brief, accurate answers. If you don't know the answer, say so.

Relevant context:
"""

In [ ]:
# Legacy: optional keyword-based context. The Gradio chat uses the retriever (RAG) via additional_context() instead.
def get_relevant_context(message):
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    print(words)
    knowledge = {}  # Not used when additional_context uses the vector-store retriever
    return [knowledge[word] for word in words if word in knowledge]   

In [ ]:
def additional_context(message):
    """Retrieve relevant embedded-systems context from the vector store for the user's message."""
    content = message if isinstance(message, str) else (getattr(message, "content", "") or str(message))
    docs = retriever.invoke(content)
    if not docs:
        return "There is no additional context relevant to the user's question."
    return "\n\n".join(doc.page_content for doc in docs)

In [ ]:
def chat(message, history):
    system_message = SYSTEM_PREFIX + additional_context(message)
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [ ]:
view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)